# U-Net source separation demo

Loads one audio file, runs the trained `VoiceSeparationUNet`, and plays the separated voice output.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import librosa
import numpy as np
import torch
from IPython.display import Audio, display

# Resolve repo root so local package imports work from notebooks/
for _root in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (_root / "source_separation").is_dir():
        sys.path.insert(0, str(_root))
        REPO_ROOT = _root
        break
else:
    raise RuntimeError("Could not find repo root containing source_separation/")

from source_separation import STFTConfig, VoiceSeparationUNet
from source_separation.stft import magnitude_to_waveform


In [5]:
# Set your paths (or use env vars)
AUDIO_PATH = Path(os.environ.get("UNET_AUDIO_PATH", ""))

AUDIO_PATH = Path("/home/luusamp/crossroads/voice-anonymization/data/mixes_eval/eval_low_snr/mix_000000.wav")
CKPT_PATH = Path(
    os.environ.get("UNET_CKPT_PATH", str(REPO_ROOT / "checkpoints/unet_run1/unet_voice_sep.pt"))
)

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(
        "Set UNET_AUDIO_PATH or edit AUDIO_PATH to a wav file path."
    )
if not CKPT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {CKPT_PATH}. Set UNET_CKPT_PATH or train first."
    )

config = STFTConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load audio and resample to training SR
y, sr = librosa.load(str(AUDIO_PATH), sr=None, mono=True)
if sr != config.sample_rate:
    y = librosa.resample(y, orig_sr=sr, target_sr=config.sample_rate)
    sr = config.sample_rate
y = y.astype(np.float32)
print(f"Loaded {AUDIO_PATH} | sr={sr} | samples={len(y)} | duration={len(y)/sr:.2f}s")

# Load model checkpoint
model = VoiceSeparationUNet().to(device)
ckpt = torch.load(str(CKPT_PATH), map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: {CKPT_PATH}")


Using device: cuda
Loaded /home/luusamp/crossroads/voice-anonymization/data/mixes_eval/eval_low_snr/mix_000000.wav | sr=16000 | samples=160000 | duration=10.00s
Loaded checkpoint: /home/luusamp/crossroads/voice-anonymization/checkpoints/unet_run1/unet_voice_sep.pt


In [6]:
def separate_voice(y: np.ndarray, model: torch.nn.Module, config: STFTConfig, device: torch.device) -> np.ndarray:
    """Run U-Net mask inference over full spectrogram using 128-frame chunks."""
    wav = torch.from_numpy(y).to(device)
    window = torch.hann_window(config.n_fft, device=device, dtype=wav.dtype)

    # Full complex STFT of mixture for phase reconstruction
    stft_full = torch.stft(
        wav,
        n_fft=config.n_fft,
        hop_length=config.hop_length,
        win_length=config.n_fft,
        window=window,
        center=config.center,
        return_complex=True,
    )  # (513, T)

    mag = stft_full.abs()[: config.n_freq_bins, :]  # (512, T)
    phase = torch.angle(stft_full)[: config.n_freq_bins, :]  # (512, T)
    T = mag.shape[-1]

    # Chunk along time into 128-frame windows
    n_frames = config.n_frames
    n_chunks = (T + n_frames - 1) // n_frames
    T_pad = n_chunks * n_frames
    if T_pad > T:
        mag = torch.nn.functional.pad(mag, (0, T_pad - T))
        phase = torch.nn.functional.pad(phase, (0, T_pad - T))

    mag_chunks = mag.view(config.n_freq_bins, n_chunks, n_frames).permute(1, 0, 2)  # (N, 512, 128)
    mag_chunks = mag_chunks.unsqueeze(1)  # (N, 1, 512, 128)

    with torch.no_grad():
        mask_chunks = model(mag_chunks)
    est_mag_chunks = (mask_chunks * mag_chunks).squeeze(1)  # (N, 512, 128)
    est_mag = est_mag_chunks.permute(1, 0, 2).reshape(config.n_freq_bins, T_pad)[:, :T]
    phase = phase[:, :T]

    y_est = magnitude_to_waveform(
        est_mag,
        phase,
        config,
        length=len(y),
        window=window,
    )
    return y_est.detach().cpu().numpy().astype(np.float32)


y_sep = separate_voice(y, model, config, device)
print(f"Separated samples: {len(y_sep)}")


Separated samples: 160000


In [7]:
print("Original")
display(Audio(y, rate=sr))
print("Separated voice (U-Net output)")
display(Audio(y_sep, rate=sr))


Original


Separated voice (U-Net output)
